# RAG generative prompts: experiments

Baseline MPLMM uses one **static** generative prompt per missing modality. Here the prompt is built per sample from the **top-K most similar training samples** (similarity on the *available* modalities; their *missing* modality is fed to the prompt through cross-attention, behind a gate that starts at 0).

All logic lives in `ragmm/` and `scripts/`; this notebook only calls the scripts. See `README.md`.

| Run | Meaning |
|---|---|
| `static` | baseline prompt, retrained through the same pipeline (fair reference) |
| `rag` | retrieval-augmented prompt, K ∈ {1, 5, 10} chosen on **validation** |
| `random` | K **random** training samples: same extra parameters, no meaningful retrieval (control) |

In [2]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "ragmm").is_dir() else Path.cwd() / "RAGPrompt"
RUNS_CSV = ROOT / "results" / "runs.csv"


def run(script, *args):
    """Run one of the scripts and stream its output into the notebook."""
    cmd = [sys.executable, str(ROOT / "scripts" / script), *map(str, args)]
    subprocess.run(cmd, check=True, cwd=ROOT, env={**os.environ, "PYTHONWARNINGS": "ignore"})


def done(name):
    return RUNS_CSV.exists() and name in set(pd.read_csv(RUNS_CSV)["run"])


print("RAGPrompt root:", ROOT)

RAGPrompt root: /home/cse/Desktop/MLProjectVidit/RAGPrompt


## 1. Memory banks, sanity checks, retrieval check (no training)

`sanity_checks.py` must end with **ALL CHECKS PASSED**. `check_retrieval.py` shows whether retrieval carries signal (real ≫ random).

In [2]:
DATASETS = ["sims", "mosi"]

for ds in DATASETS:
    run("build_bank.py", "--dataset", ds)
    run("sanity_checks.py", "--dataset", ds)
    run("check_retrieval.py", "--dataset", ds)

Saved /home/cse/Desktop/MLProjectVidit/RAGPrompt/banks/sims_bank.pt
  samples: 1368  groups: 60  size: 134.0 MB
  l: key (1368, 768)  value (1368, 16, 768)
  a: key (1368, 33)  value (1368, 16, 33)
  v: key (1368, 709)  value (1368, 16, 709)
[PASS] bank size equals training split  (1368 vs 1368)
[PASS] bank entry i is training sample i (labels match)
[PASS] no retrieval from the sample's own video/dialogue  (0 leaks)
[PASS] no sample retrieves itself  (0 self-hits)
[PASS] --exclude self: no sample retrieves itself  (0 self-hits)
Missing key(s) in state_dict :proj_l.weight
Missing key(s) in state_dict :proj_a.weight
Missing key(s) in state_dict :proj_v.weight
Missing key(s) in state_dict :out_layer.weight
Missing key(s) in state_dict :out_layer.bias
[PASS] only rag_* parameters are new  (25 new tensors)
[PASS] static (RAG off) == baseline PromptModel  (max diff 0.00e+00)
[PASS] RAG with gate 0 == baseline PromptModel  (max diff 0.00e+00)
[PASS] RAG with gate open != baseline (retrieval 

## 2. Training: static vs RAG vs random (seed 1111)

About 5 min per CH-SIMS run and 8 min per MOSI run. Runs already in `results/runs.csv` are skipped.

In [3]:
SEED = 1111
GRID = [("static", None), ("rag", 1), ("rag", 5), ("rag", 10), ("random", 5)]

for ds in DATASETS:
    for mode, k in GRID:
        name = f"{ds}_static_seed{SEED}" if mode == "static" else f"{ds}_{mode}_k{k}_seed{SEED}"
        if done(name):
            print("[skip]", name)
            continue
        args = ["--dataset", ds, "--mode", mode, "--seed", SEED] + (["--k", k] if k else [])
        run("train_rag.py", *args)

=== sims_static_seed1111 ===
Missing key(s) in state_dict :proj_l.weight
Missing key(s) in state_dict :proj_a.weight
Missing key(s) in state_dict :proj_v.weight
Missing key(s) in state_dict :out_layer.weight
Missing key(s) in state_dict :out_layer.bias
trainable parameters: 442,168  (new rag_*: 0)
epoch   1 |   8.4s | train 0.5984 | valid 0.5905 *
epoch   2 |   8.2s | train 0.5906 | valid 0.5884 *
epoch   3 |   8.2s | train 0.5896 | valid 0.5867 *
epoch   4 |   8.3s | train 0.5823 | valid 0.5743 *
epoch   5 |   8.5s | train 0.5537 | valid 0.5419 *
epoch   6 |   8.7s | train 0.5157 | valid 0.5462
epoch   7 |   9.1s | train 0.5012 | valid 0.5237 *
epoch   8 |   9.7s | train 0.4823 | valid 0.5082 *
epoch   9 |  10.7s | train 0.4800 | valid 0.5202
epoch  10 |  19.3s | train 0.4788 | valid 0.4767 *
epoch  11 |  39.3s | train 0.4405 | valid 0.4903
epoch  12 |  43.8s | train 0.4413 | valid 0.5012
epoch  13 |  44.5s | train 0.4425 | valid 0.4821
epoch  14 |  45.0s | train 0.4512 | valid 0.4795

## 3. Results

For RAG, K is chosen on **validation** Avg. ACC; only then is its **test** row shown. Δ = difference from `static`.

In [4]:
CASES = ["{a}", "{v}", "{t}", "{a,v}", "{a,t}", "{v,t}", "Avg."]
runs = pd.read_csv(RUNS_CSV)

rows = []
for ds, group in runs.groupby("dataset", sort=False):
    rag = group[group["mode"] == "rag"]
    picks = {
        "static": group[group["mode"] == "static"],
        "rag (K by valid)": rag.loc[[rag["valid Avg. ACC"].idxmax()]] if len(rag) else rag,
        "random": group[group["mode"] == "random"],
    }
    for label, picked in picks.items():
        if len(picked):
            r = picked.iloc[0]
            row = {"Dataset": ds, "Run": label, "K": r["k"]}
            row.update({f"{c} ACC/F1": f"{r[f'test {c} ACC']:.2f} / {r[f'test {c} F1']:.2f}" for c in CASES})
            row["Avg. ACC"], row["Avg. F1"] = r["test Avg. ACC"], r["test Avg. F1"]
            rows.append(row)

table = pd.DataFrame(rows)
static = table[table["Run"] == "static"].set_index("Dataset")[["Avg. ACC", "Avg. F1"]]
table["Δ Avg. ACC"] = table.apply(lambda r: r["Avg. ACC"] - static.loc[r["Dataset"], "Avg. ACC"], axis=1).round(2)
table["Δ Avg. F1"] = table.apply(lambda r: r["Avg. F1"] - static.loc[r["Dataset"], "Avg. F1"], axis=1).round(2)

pd.set_option("display.max_columns", None)
display(table)
table.to_csv(ROOT / "results" / "comparison_table.csv", index=False)

,Dataset,Run,K,{a} ACC/F1,{v} ACC/F1,{t} ACC/F1,"{a,v} ACC/F1","{a,t} ACC/F1","{v,t} ACC/F1",Avg. ACC/F1,Avg. ACC,Avg. F1,Δ Avg. ACC,Δ Avg. F1
0,sims,static,NaN,64.33 / 61.97,69.58 / 69.99,72.87 / 73.70,70.46 / 71.44,75.71 / 76.38,76.81 / 77.40,71.63 / 71.81,71.63,71.81,0.00,0.00
1,sims,rag (K by valid),5.0,66.96 / 59.39,76.15 / 73.17,77.90 / 77.04,73.52 / 69.32,77.46 / 76.78,78.34 / 76.90,75.05 / 72.10,75.05,72.10,3.42,0.29
2,sims,random,5.0,67.18 / 58.70,73.30 / 68.64,72.43 / 73.20,70.46 / 59.66,72.87 / 73.65,77.68 / 77.85,72.32 / 68.62,72.32,68.62,0.69,-3.19
3,mosi,static,NaN,49.56 / 42.56,46.06 / 33.34,73.18 / 73.23,45.77 / 33.18,73.32 / 73.33,71.87 / 71.90,59.96 / 54.59,59.96,54.59,0.00,0.00
4,mosi,rag (K by valid),10.0,53.94 / 54.04,56.85 / 56.78,72.45 / 72.37,46.21 / 33.62,73.32 / 73.40,73.47 / 73.54,62.71 / 60.62,62.71,60.62,2.75,6.03
5,mosi,random,5.0,55.54 / 52.27,51.60 / 47.83,74.05 / 73.88,57.87 / 57.40,73.76 / 73.60,72.89 / 72.89,64.29 / 62.98,64.29,62.98,4.33,8.39


## 4. Remaining datasets: IEMOCAP and MOSEI (RAG, K = 1, 5, 10)

Same method and settings as CH-SIMS and MOSI. Each run takes about **70–80 min** (6 runs ≈ 7–8 h). Finished runs are skipped, so the cell can be stopped and re-run safely; the run that was interrupted starts again from scratch.

In [4]:
SEED = 1111

for ds in ["iemocap", "mosei"]:
    bank_file = ROOT / "banks" / ("iemocap_a350_bank.pt" if ds == "iemocap" else f"{ds}_bank.pt")
    if not bank_file.exists():
        run("build_bank.py", "--dataset", ds)
        run("sanity_checks.py", "--dataset", ds)
    for k in [1, 5, 10]:
        name = f"{ds}_rag_k{k}_seed{SEED}"
        if done(name):
            print("[skip]", name)
            continue
        run("train_rag.py", "--dataset", ds, "--mode", "rag", "--k", k, "--seed", SEED)

=== iemocap_rag_k1_seed1111 ===
Missing key(s) in state_dict :proj_l.weight
Missing key(s) in state_dict :proj_a.weight
Missing key(s) in state_dict :proj_v.weight
Missing key(s) in state_dict :out_layer.weight
Missing key(s) in state_dict :out_layer.bias
trainable parameters: 374,911  (new rag_*: 59,125)
epoch   1 |  82.7s | train 1.3268 | valid 1.2601 * | gates [0.013, -0.011, 0.004]
epoch   2 |  23.1s | train 1.2196 | valid 1.1907 * | gates [0.015, -0.044, 0.004]
epoch   3 |  24.8s | train 1.1479 | valid 1.1398 * | gates [0.002, -0.063, -0.0]
epoch   4 |  41.6s | train 1.1279 | valid 1.1104 * | gates [-0.001, -0.049, 0.002]
epoch   5 | 107.2s | train 1.0654 | valid 1.1590 | gates [0.003, -0.073, 0.005]
epoch   6 | 111.7s | train 1.0279 | valid 1.1035 * | gates [0.005, -0.062, 0.008]
epoch   7 | 112.4s | train 0.9955 | valid 1.0964 * | gates [0.002, -0.063, -0.003]
epoch   8 | 112.2s | train 0.9871 | valid 1.1111 | gates [0.0, -0.079, -0.002]
epoch   9 | 111.6s | train 0.9594 | valid

## 5. Final table: RAG on all four datasets

Test ACC / F1 for every K. `selected` marks the K with the best **validation** Avg. ACC for each dataset (the one to report); test scores play no part in choosing it.

In [5]:
CASES = ["{a}", "{v}", "{t}", "{a,v}", "{a,t}", "{v,t}", "Avg."]
NAMES = {"mosi": "MOSI", "iemocap": "IEMOCAP", "sims": "CH-SIMS", "mosei": "MOSEI"}

runs = pd.read_csv(RUNS_CSV)
rag = runs[runs["mode"] == "rag"].copy()
rag["selected"] = rag["valid Avg. ACC"] == rag.groupby("dataset")["valid Avg. ACC"].transform("max")

rows = []
for ds, label in NAMES.items():
    for _, r in rag[rag["dataset"] == ds].sort_values("k").iterrows():
        row = {"Dataset": label, "K": int(r["k"]), "selected": "✓" if r["selected"] else ""}
        for c in CASES:
            row[f"{c} ACC"] = r[f"test {c} ACC"]
            row[f"{c} F1"] = r[f"test {c} F1"]
        rows.append(row)

final_table = pd.DataFrame(rows)
final_table.to_csv(ROOT / "results" / "final_rag_table.csv", index=False)
pd.set_option("display.max_columns", None)
display(final_table)

,Dataset,K,selected,{a} ACC,{a} F1,{v} ACC,{v} F1,{t} ACC,{t} F1,"{a,v} ACC","{a,v} F1","{a,t} ACC","{a,t} F1","{v,t} ACC","{v,t} F1",Avg. ACC,Avg. F1
0,MOSI,1,,54.96,52.38,46.79,34.14,66.18,66.21,46.65,34.06,63.85,63.65,68.22,68.30,57.77,53.12
1,MOSI,5,,53.50,53.57,50.15,47.34,76.38,75.72,46.94,39.97,75.80,75.20,72.89,72.85,62.61,60.77
2,MOSI,10,✓,53.94,54.04,56.85,56.78,72.45,72.37,46.21,33.62,73.32,73.40,73.47,73.54,62.71,60.62
3,IEMOCAP,1,,51.89,50.72,56.25,53.95,56.25,55.38,62.69,61.54,61.74,60.23,61.36,59.79,58.36,56.94
4,IEMOCAP,5,✓,52.08,50.35,55.87,54.25,58.71,58.48,64.58,63.95,64.02,62.86,62.50,61.31,59.63,58.53
5,IEMOCAP,10,,49.43,46.29,57.20,54.28,58.33,57.74,64.02,62.89,61.74,60.28,64.96,64.13,59.28,57.60
6,CH-SIMS,1,,60.39,60.19,70.24,70.24,77.02,77.18,70.68,71.18,75.27,75.68,76.37,76.98,71.66,71.91
7,CH-SIMS,5,✓,66.96,59.39,76.15,73.17,77.90,77.04,73.52,69.32,77.46,76.78,78.34,76.90,75.05,72.10
8,CH-SIMS,10,,67.61,58.05,73.74,71.18,75.27,76.06,73.09,68.81,76.81,77.32,76.15,76.72,73.78,71.36
9,MOSEI,1,,65.13,65.70,64.63,64.75,77.06,77.65,65.30,66.32,76.87,77.54,76.98,77.60,71.00,71.59
